# AI-Powered Personal Spending & Financial Behavior Coach
## Notebook 06 — LLM Integration + Hugging Face Deployment (FINAL)

### Objective
Deploy a live AI coach that:
1) Uses the best ML model (Gradient Boosting) to predict "High Spender" risk (probability)
2) Applies a balanced decision threshold of **0.60**
3) Uses an LLM to generate clear coaching language
4) Runs as a Hugging Face Space (Streamlit)

### What Changed (Important)
During modeling, the best performer was **Gradient Boosting** (ROC AUC ~0.9566).
However, the saved artifact `final_model.joblib` initially contained a tuned Random Forest.

This notebook overwrites `final_model.joblib` with the **Gradient Boosting pipeline** to ensure deployment uses the best model.

### Compliance Note
Educational insights only — not financial advice.

## 0) Imports

We use:
- pandas/numpy for data handling
- joblib for model serialization
- scikit-learn pipelines for preprocessing + modeling
- pathlib to write Hugging Face app files

In [27]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import joblib

## 1) Load Data and Check Artifacts

We require:
- cleaned_transactions.csv (to populate UI dropdowns and build model input)
- final_model.joblib (trained pipeline used by Streamlit app)

If `final_model.joblib` currently contains RandomForest, we will overwrite it with Gradient Boosting.

In [30]:
df = pd.read_csv("cleaned_transactions.csv")

print("cleaned_transactions.csv ✅")
print("Rows:", len(df), "| Columns:", df.columns.tolist())
print("final_model.joblib exists?", Path("final_model.joblib").exists())

cleaned_transactions.csv ✅
Rows: 50000 | Columns: ['customer_id', 'name', 'surname', 'gender', 'birthdate', 'transaction_amount', 'date', 'merchant_name', 'category', 'merchant_name_clean', 'year', 'month', 'day', 'weekday', 'age', 'age_group']
final_model.joblib exists? True


## 2) Confirm the Model Currently Saved in final_model.joblib

In our case, the saved pipeline was found to contain:
- RandomForestClassifier(max_depth=12, min_samples_split=10, n_estimators=400, ...)

Because Gradient Boosting performed best, we overwrite the artifact with Gradient Boosting.

In [33]:
if Path("final_model.joblib").exists():
    current_model = joblib.load("final_model.joblib")
    print("Saved pipeline type:", type(current_model))
    print("Saved estimator:", current_model.named_steps["model"])
else:
    print("final_model.joblib not found — we will create it.")

Saved pipeline type: <class 'sklearn.pipeline.Pipeline'>
Saved estimator: RandomForestClassifier(max_depth=12, min_samples_split=10, n_estimators=400,
                       n_jobs=-1, random_state=42)


## 3) Create + Save the Gradient Boosting Pipeline (Overwrite final_model.joblib)

We rebuild the Gradient Boosting pipeline using the same feature strategy as the modeling notebook:
- Predict high_spender = top 25% transaction_amount
- Features: age, gender, category, and merchant (if available)
- Preprocessing: StandardScaler for age + OneHotEncoder for categorical variables

Then we overwrite `final_model.joblib` with this pipeline so deployment uses the best model.

In [36]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier

# Target
threshold = df["transaction_amount"].quantile(0.75)
df["high_spender"] = (df["transaction_amount"] >= threshold).astype(int)

# Features
merchant_col = "merchant_name_clean" if "merchant_name_clean" in df.columns else ("merchant_name" if "merchant_name" in df.columns else None)

feature_cols = ["age", "gender", "category"]
if merchant_col:
    feature_cols.append(merchant_col)

X = df[feature_cols].copy()
y = df["high_spender"].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_features = ["age"]
categorical_features = [c for c in feature_cols if c not in numeric_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

gb_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", GradientBoostingClassifier(random_state=42))
])

gb_pipeline.fit(X_train, y_train)

joblib.dump(gb_pipeline, "final_model.joblib")
print("✅ Overwrote final_model.joblib with Gradient Boosting pipeline")

✅ Overwrote final_model.joblib with Gradient Boosting pipeline


## 4) Confirm Saved Model is Gradient Boosting

We reload `final_model.joblib` and confirm the estimator inside the pipeline.

In [39]:
model = joblib.load("final_model.joblib")
print("Saved pipeline:", type(model))
print("Saved estimator:", model.named_steps["model"])

Saved pipeline: <class 'sklearn.pipeline.Pipeline'>
Saved estimator: GradientBoostingClassifier(random_state=42)


## 5) Balanced Decision Threshold (0.60) + Risk Buckets

We deploy a balanced threshold of **0.60**.
- This reduces false positives compared to 0.50
- Still supports high recall for coaching use

Risk buckets shown in the UI:
- Low: < 0.35
- Moderate: 0.35–0.65
- High: >= 0.65

In [42]:
DECISION_THRESHOLD = 0.60

def risk_bucket(p: float) -> str:
    if p < 0.35:
        return "Low"
    if p < 0.65:
        return "Moderate"
    return "High"

def is_flagged(p: float, threshold: float = DECISION_THRESHOLD) -> int:
    return int(p >= threshold)

## 6) Prompt Template (Safe Coaching)

The LLM message is:
- educational
- non-prescriptive
- supportive
- includes a clear disclaimer

We pass:
- probability score
- risk bucket
- flagged decision at threshold 0.60

In [45]:
def build_prompt(age, gender, category, merchant, risk_prob, flagged):
    bucket = risk_bucket(risk_prob)
    flag_text = "YES" if flagged == 1 else "NO"

    return f"""
You are a friendly financial behavior coach. Explain patterns in clear, supportive language.

Rules:
- Educational insights only.
- Do NOT give regulated financial advice.
- Do NOT promise outcomes.
- Keep it concise and actionable.
- Use a supportive, non-judgmental tone.

Context:
- Age: {age}
- Gender: {gender}
- Category: {category}
- Merchant: {merchant}
- High-spender probability (model): {risk_prob:.2f}
- Risk bucket: {bucket}
- Flagged as high spender at threshold 0.60? {flag_text}

Write:
1) A 2–3 sentence plain-English summary of what this suggests.
2) Three bullet suggestions starting with "Consider..."
3) One reflective question.
4) End with: "Educational only, not financial advice."
""".strip()

## 7) LLM Generation via Hugging Face Inference API

Deployment setup:
- Add HF_TOKEN in Hugging Face Space Settings → Secrets
- The app calls `google/flan-t5-large` by default

If HF_TOKEN is missing, the app displays a friendly configuration message.

In [48]:
import requests

def hf_inference_generate(prompt: str, model_id: str = "google/flan-t5-large", timeout: int = 60) -> str:
    token = os.getenv("HF_TOKEN")
    if not token:
        return "LLM not configured. Add HF_TOKEN in Hugging Face Space Secrets."

    url = f"https://api-inference.huggingface.co/models/{model_id}"
    headers = {"Authorization": f"Bearer {token}"}
    payload = {"inputs": prompt, "parameters": {"max_new_tokens": 220, "do_sample": False}}

    try:
        r = requests.post(url, headers=headers, json=payload, timeout=timeout)
        r.raise_for_status()
        data = r.json()

        if isinstance(data, list) and len(data) > 0 and "generated_text" in data[0]:
            return data[0]["generated_text"]

        return str(data)[:1500]

    except Exception as e:
        return f"LLM request failed: {e}"

## 8) End-to-End Test

We test one example input:
- compute model probability
- apply threshold (0.60)
- generate the LLM prompt

In [51]:
categories = sorted(df["category"].dropna().unique().tolist())
genders = sorted(df["gender"].dropna().unique().tolist())
merchants = sorted(df[merchant_col].dropna().unique().tolist()) if merchant_col else ["N/A"]

age = int(np.nanmedian(df["age"]))
gender = "Unknown" if "Unknown" in genders else genders[0]
category = categories[0]
merchant = merchants[0] if merchants else "N/A"

row = {"age": age, "gender": gender, "category": category}
if merchant_col:
    row[merchant_col] = merchant

X_one = pd.DataFrame([row])

risk_prob = float(model.predict_proba(X_one)[:, 1][0])
flagged = is_flagged(risk_prob, DECISION_THRESHOLD)
prompt = build_prompt(age, gender, category, merchant, risk_prob, flagged)

print("Probability:", round(risk_prob, 4))
print("Flagged @ 0.60:", flagged)
print("Risk bucket:", risk_bucket(risk_prob))
print("\nPrompt preview:\n", prompt[:650], "...")

Probability: 0.0084
Flagged @ 0.60: 0
Risk bucket: Low

Prompt preview:
 You are a friendly financial behavior coach. Explain patterns in clear, supportive language.

Rules:
- Educational insights only.
- Do NOT give regulated financial advice.
- Do NOT promise outcomes.
- Keep it concise and actionable.
- Use a supportive, non-judgmental tone.

Context:
- Age: 49
- Gender: Unknown
- Category: Clothing
- Merchant: abbott and sons
- High-spender probability (model): 0.01
- Risk bucket: Low
- Flagged as high spender at threshold 0.60? NO

Write:
1) A 2–3 sentence plain-English summary of what this suggests.
2) Three bullet suggestions starting with "Consider..."
3) One reflective question.
4) End with: "Educational  ...


## 9) Create Hugging Face Streamlit App (app.py)

The app:
- loads Gradient Boosting pipeline from final_model.joblib
- predicts probability
- applies balanced threshold 0.60
- generates an LLM coaching message
- shows transparency (prompt expander)

In [56]:
app_code = r'''
import os
import pandas as pd
import joblib
import streamlit as st
import requests

st.set_page_config(page_title="AI Spending Coach", page_icon="💳", layout="centered")

DECISION_THRESHOLD = 0.60  # balanced

@st.cache_resource
def load_artifacts():
    model = joblib.load("final_model.joblib")
    df = pd.read_csv("cleaned_transactions.csv")
    merchant_col = "merchant_name_clean" if "merchant_name_clean" in df.columns else ("merchant_name" if "merchant_name" in df.columns else None)
    return model, df, merchant_col

def risk_bucket(p: float) -> str:
    if p < 0.35:
        return "Low"
    if p < 0.65:
        return "Moderate"
    return "High"

def is_flagged(p: float, threshold: float = DECISION_THRESHOLD) -> int:
    return int(p >= threshold)

def build_prompt(age, gender, category, merchant, risk_prob, flagged):
    bucket = risk_bucket(risk_prob)
    flag_text = "YES" if flagged == 1 else "NO"
    return f"""
You are a friendly financial behavior coach. Explain patterns in clear, supportive language.

Rules:
- Educational insights only.
- Do NOT give regulated financial advice.
- Do NOT promise outcomes.
- Keep it concise and actionable.
- Use a supportive, non-judgmental tone.

Context:
- Age: {age}
- Gender: {gender}
- Category: {category}
- Merchant: {merchant}
- High-spender probability (model): {risk_prob:.2f}
- Risk bucket: {bucket}
- Flagged as high spender at threshold 0.60? {flag_text}

Write:
1) A 2–3 sentence plain-English summary of what this suggests.
2) Three bullet suggestions starting with "Consider..."
3) One reflective question.
4) End with: "Educational only, not financial advice."
""".strip()

def hf_inference_generate(prompt: str, model_id: str = "google/flan-t5-large", timeout: int = 60) -> str:
    token = os.getenv("HF_TOKEN")
    if not token:
        return "LLM not configured. Add HF_TOKEN in Hugging Face Space Secrets."

    url = f"https://api-inference.huggingface.co/models/{model_id}"
    headers = {"Authorization": f"Bearer {token}"}
    payload = {"inputs": prompt, "parameters": {"max_new_tokens": 220, "do_sample": False}}

    try:
        r = requests.post(url, headers=headers, json=payload, timeout=timeout)
        r.raise_for_status()
        data = r.json()
        if isinstance(data, list) and len(data) > 0 and "generated_text" in data[0]:
            return data[0]["generated_text"]
        return str(data)[:1500]
    except Exception as e:
        return f"LLM request failed: {e}"

# -------------------- UI --------------------
st.title("💳 AI Personal Spending Coach")
st.caption("Educational insights only — not financial advice.")

model, df, merchant_col = load_artifacts()

categories = sorted(df["category"].dropna().unique().tolist())
genders = sorted(df["gender"].dropna().unique().tolist())
merchants = sorted(df[merchant_col].dropna().unique().tolist()) if merchant_col else ["N/A"]

st.subheader("Enter your context")
age = st.slider("Age", min_value=18, max_value=85, value=30)

default_gender = "Unknown" if "Unknown" in genders else genders[0]
gender = st.selectbox("Gender", options=genders, index=genders.index(default_gender))

category = st.selectbox("Category", options=categories, index=0)
merchant = st.selectbox("Merchant", options=merchants, index=0)

st.markdown(f"**Decision threshold:** `{DECISION_THRESHOLD}` (balanced)")

if st.button("Generate my coaching insight"):
    row = {"age": age, "gender": gender, "category": category}
    if merchant_col:
        row[merchant_col] = merchant

    X_one = pd.DataFrame([row])
    risk_prob = float(model.predict_proba(X_one)[:, 1][0])
    flagged = is_flagged(risk_prob, DECISION_THRESHOLD)
    bucket = risk_bucket(risk_prob)

    st.markdown("### Prediction")
    st.metric("High-spender probability", f"{risk_prob:.2f}")
    st.write(f"Risk bucket: **{bucket}**")
    st.write(f"Flagged as high spender @ 0.60? **{'Yes' if flagged==1 else 'No'}**")

    prompt = build_prompt(age, gender, category, merchant, risk_prob, flagged)

    st.markdown("### AI Coach Message")
    coaching_text = hf_inference_generate(prompt, model_id="google/flan-t5-large")
    st.write(coaching_text)

    with st.expander("Show prompt (transparency)"):
        st.code(prompt)
'''

Path("app.py").write_text(app_code, encoding="utf-8")
print("✅ Wrote app.py")

✅ Wrote app.py


## 10) Create requirements.txt

In [59]:
reqs = """streamlit
pandas
numpy
scikit-learn
joblib
requests
"""
Path("requirements.txt").write_text(reqs, encoding="utf-8")
print("✅ Wrote requirements.txt")

✅ Wrote requirements.txt


## 11) Verify Deployment Package

Upload these files to Hugging Face Space:
- app.py
- requirements.txt
- final_model.joblib (Gradient Boosting pipeline)
- cleaned_transactions.csv

In [62]:
for f in ["app.py", "requirements.txt", "final_model.joblib", "cleaned_transactions.csv"]:
    print(f, "✅" if Path(f).exists() else "❌ NOT FOUND")

app.py ✅
requirements.txt ✅
final_model.joblib ✅
cleaned_transactions.csv ✅


## 12) Hugging Face Deployment Steps

1) Create a new Hugging Face Space  
   - SDK: Streamlit

2) Upload:
   - app.py
   - requirements.txt
   - final_model.joblib
   - cleaned_transactions.csv

3) Add Secret:
   - Settings → Secrets → HF_TOKEN = your Hugging Face token

4) When build completes → open your Space URL → test.

✅ Your AI coach is live.